In [23]:
from collections import Counter
import re

def find_words(text): return re.findall(r'\w+', text.lower())

VOCAB = Counter(find_words(open("big.txt").read()))
INDIR_BIGRAM = {}
DIR_BIGRAM = {}

TOP_K = 20

In [24]:
# Read first file
data = []

with open("data/bigrams.txt", "rb") as f:
	for byte_line in f:
		try:
			line = byte_line.decode("utf-8")
			data.append(line.strip())
		except UnicodeDecodeError:
			pass
for line in data:
	occur, w1, w2 = line.split()
	occur = int(occur)
	w1, w2 = w1.lower(), w2.lower()

	if w1 not in DIR_BIGRAM.keys():
		DIR_BIGRAM[w1] = {}

	if w2 not in INDIR_BIGRAM.keys():
		INDIR_BIGRAM[w2] = {}

	DIR_BIGRAM[w1][w2] = occur
	INDIR_BIGRAM[w2][w1] = occur

In [25]:
# Read second file
data = []
with open("data/coca_all_links.txt", "rb") as f:
	for byte_line in f:
		try:
			line = byte_line.decode("utf-8")
			data.append(line.strip())
		except UnicodeDecodeError:
			pass


for line in data:
	line = line.split()
	occur, w1, w2 = int(line[0]), line[1].lower(), line[2].lower()


	if w1 not in DIR_BIGRAM.keys():
		DIR_BIGRAM[w1] = {}

	if w2 not in DIR_BIGRAM[w1].keys():
		DIR_BIGRAM[w1][w2] = 0

	DIR_BIGRAM[w1][w2] += occur



	if w2 not in INDIR_BIGRAM.keys():
		INDIR_BIGRAM[w2] = {}

	if w1 not in INDIR_BIGRAM[w2].keys():
		INDIR_BIGRAM[w2][w1] = 0

	INDIR_BIGRAM[w2][w1] += occur


In [26]:
def calc_probs(vocab):
	for key in vocab.keys():
		words = vocab[key].items()
		total = sum([num for _, num in words])
		vocab[key] = list(map(lambda x: (x[0], x[1]/total), words))

calc_probs(DIR_BIGRAM)
calc_probs(INDIR_BIGRAM)

In [28]:
max(DIR_BIGRAM["a"], key=lambda x: x[1])

('lot', 0.021956376402660804)

In [67]:
def P(word):
	return VOCAB[word] / sum(VOCAB.values())


def is_known(word):
	return [w for w in word if w in VOCAB]


def edit1(word):
    letters    = 'abcdefghijklmnopqrstuvwxyz'
    splits     = [(word[:i], word[i:])    for i in range(len(word) + 1)]
    deletes    = [L + R[1:]               for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R)>1]
    replaces   = [L + c + R[1:]           for L, R in splits if R for c in letters]
    inserts    = [L + c + R               for L, R in splits for c in letters]
    return set(deletes + transposes + replaces + inserts)


def edit2(word, eds=None):
	if eds is None:
		eds = edit1(word)
	return (e2 for e1 in eds for e2 in edit1(e1))


def norvig_solution(word):
	eds1 = edit1(word)
	eds2 = edit2(word, eds1)
	candidates = is_known([word]) or is_known(eds1) or is_known(eds2)
	print("fdf")
	print(candidates)
	# TODO keep more candidates for precision (???)
	return max(candidates, key=P)

In [59]:
import numpy as np

def levenstein_distance(word1, word2):
	w1_len = len(word1)
	w2_len = len(word2)
	
	lev = np.zeros((w1_len, w2_len), dtype=int)
	
	lev[:, 0] = np.arange(w1_len)
	lev[0, :] = np.arange(w2_len)
	
	for i in range(1,w1_len):
		for j in range(1,w2_len):
			m = 0
			if word1[i] == word2[j]:
				m = 1
			lev[i,j] = min(lev[i-1, j] + 1, lev[i-1, j-1] + m, lev[i-1, j] + 1)
			
	return lev[w1_len-1,w2_len-1].item()

In [48]:
def look_forward(word, next_word):
	prev_words = INDIR_BIGRAM[next_word]

	max_candidates = min(len(INDIR_BIGRAM), TOP_K)

	candidates = prev_words[:max_candidates]
	return candidates

In [35]:
def look_behind(word, prev_word):
	next_words = DIR_BIGRAM[prev_word]

	max_candidates = min(len(DIR_BIGRAM), TOP_K)

	candidates = next_words[:max_candidates]

	return candidates

In [61]:
def correction(word, candidates):
	dists = [levenstein_distance(word, c) for c in candidates]
	c = list(zip(candidates, dists))
	print(c)
	sorted_candidates = c.sort(key=lambda x: x[1], reverse=True)
	return sorted_candidates[0]

In [65]:
def text_correction(text):
	text = text.split()

	output = []

	for idx, word in enumerate(text):
		print(word)
		candidates = []

		# Base case - Norvig candidates
		norvig_candidates = norvig_solution(word)
		candidates.extend(norvig_candidates)
		print(candidates)

		# Look forward
		if idx != len(text)-1:
			forward_candidates = look_forward(word, text[idx+1])
			candidates.extend(forward_candidates)

		# Look Behind
		if idx != 0:
			behind_candidates = look_behind(word, text[idx-1])
			candidates.extend(behind_candidates)

		# TODO consider levenstein distance
		corrected = correction(word, candidates)
		output.append(corrected)

	return " ".join(output)

In [68]:
text = "Hello World this is my final message".lower()
output = text_correction(text)
print("Corrected text:\n\t")
print(output)

hello
fdf
['hello']
['h', 'e', 'l', 'l', 'o']
[('h', 4), ('e', 4), ('l', 4), ('l', 4), ('o', 4), (('a', 0.04601097214306444), 3), (('about', 0.0009160935254565607), 3), (('academic', 0.0007377699766081462), 3), (('actual', 0.0007482595971286412), 3), (('adult', 0.0006398668517501932), 3), (('advertising', 9.440658468445473e-05), 3), (('african', 0.00011538582572544468), 3), (('after', 0.007577002555970867), 3), (('alien', 0.00030419899509435413), 3), (('all', 0.00016783392832791952), 3), (('alternate', 9.440658468445473e-05), 3), (('alternative', 0.00012937198641943797), 3), (('american', 0.0003391643968293374), 3), (('among', 8.391696416395976e-05), 3), (('ancient', 0.0011958167393364267), 3), (('and', 0.0067762948562397505), 3), (('animal', 0.0002832197540533642), 3), (('annual', 0.00015734430780742456), 3), (('another', 0.003255278901526939), 3), (('any', 0.00027972321387986586), 3)]


TypeError: 'NoneType' object is not subscriptable

In [ ]:
# Your code here